In [1]:
def assign_buckets(kmer: int, k: int):
    
    buckets = [0]* k
    num_A = [0] * k
    val = [0] * k
    mu = [0] * k

    # Initialize
    mask = 3 << ((k - 1) * 2)  # 3lu << ((n-1)<<1)
    p = 1 << ((k - 1) * 2)     # 4^(n-1)
    cur = kmer & mask

    val[0] = kmer - cur
    mu[0] = p + ((cur >> 2) * (k - 1)) if cur else val[0]
    sum_mu = mu[0]
    # print(f"num_A: {num_A}, mask {bin(mask)}:, cur: {bin(cur)}, p:{bin(p)}, val: {val}")
    
    # Iterate through the k-mer
    for i in range(1, k):
        num_A[i] = num_A[i - 1] + (0 if cur else 1) # add to the number of As if the current base is A, else don't

        mask >>= 2               # Move mask to the right by 2 bits
        cur = kmer & mask           # Extract the current base
        p >>= 2                  # Reduce power by 4
        val[i] = val[i - 1] - cur
        mu[i] = p + ((cur >> 2) * (k - i - 1)) if cur else val[i]
        sum_mu += mu[i]
        
        # print(f"num_A: {num_A}, mask {bin(mask)}:, cur: {bin(cur)}, p:{bin(p)}, val[i]: {val}")

    # Initialize positions for storing in the buckets
    mask = 3 << ((k - 1) * 2)
    j = 0
    
    
    # Store the bucket values for that kmer
    for i in range(k):
        cur = kmer & mask
        mask >>= 2

        p = sum_mu - mu[i] + val[i] - num_A[i] * cur + 1 + num_A[i]

        buckets[j] = p
        j += 1

    return buckets



In [2]:
def kmer_to_binary(kmer: str) -> int:
    """Convert a DNA k-mer string to a binary representation using 2 bits per base."""
    base_to_bits = {'A': 0b00, 'C': 0b01, 'G': 0b10, 'T': 0b11}
    
    binary = 0
    for base in kmer:
        binary = (binary << 2) | base_to_bits[base]
    
    return binary

def binary_to_kmer(binary: int, k: int) -> str:
    """Convert a binary representation to a DNA k-mer string."""
    bits_to_base = ['A', 'C', 'G', 'T']
    
    kmer = []
    for _ in range(k):
        # Extract the last 2 bits
        base = binary & 0b11  
        kmer.append(bits_to_base[base])
        binary >>= 2

    # Reverse since the bases are extracted in reverse order
    return ''.join(reversed(kmer))

In [3]:
print(kmer_to_binary('AGG'))

10


In [8]:
print(len("AGCGGTACGTAGCTGACGT"))
print(kmer_to_binary("AGCGGTACGTAGCTGACGT"))
print(assign_buckets(kmer_to_binary("AGCGGTACGTAGCTGACGT"), len("AGCGGTACGTAGCTGACGT")))

19
41547505179
[238258108556, 47877379752, 215381104296, 227729135272, 235782198952, 237342480040, 238258108557, 238236915369, 238248449705, 238254544553, 238258108558, 238257944234, 238258089642, 238258095018, 238258106282, 238258108559, 238258108483, 238258108525, 238258108547]


In [5]:
import itertools
def generate_acgt_strings(k):
    """Generate all strings of length k composed of A, C, G, T."""
    for combo in itertools.product('ACGT', repeat=k):
        yield ''.join(combo)

In [6]:
for i, s in enumerate(generate_acgt_strings(3)):
    kmer_bin = kmer_to_binary(s)
    bucks = assign_buckets(kmer_bin, 3)
    print(i, s, bucks, sum(bucks))

0 AAA [1, 2, 3] 6
1 AAC [4, 5, 3] 12
2 AAG [6, 7, 3] 16
3 AAT [8, 9, 3] 20
4 ACA [10, 2, 11] 23
5 ACC [12, 5, 11] 28
6 ACG [13, 7, 11] 31
7 ACT [14, 9, 11] 34
8 AGA [15, 2, 16] 33
9 AGC [17, 5, 16] 38
10 AGG [18, 7, 16] 41
11 AGT [19, 9, 16] 44
12 ATA [20, 2, 21] 43
13 ATC [22, 5, 21] 48
14 ATG [23, 7, 21] 51
15 ATT [24, 9, 21] 54
16 CAA [1, 25, 26] 52
17 CAC [4, 27, 26] 57
18 CAG [6, 28, 26] 60
19 CAT [8, 29, 26] 63
20 CCA [10, 25, 30] 65
21 CCC [12, 27, 30] 69
22 CCG [13, 28, 30] 71
23 CCT [14, 29, 30] 73
24 CGA [15, 25, 31] 71
25 CGC [17, 27, 31] 75
26 CGG [18, 28, 31] 77
27 CGT [19, 29, 31] 79
28 CTA [20, 25, 32] 77
29 CTC [22, 27, 32] 81
30 CTG [23, 28, 32] 83
31 CTT [24, 29, 32] 85
32 GAA [1, 33, 34] 68
33 GAC [4, 35, 34] 73
34 GAG [6, 36, 34] 76
35 GAT [8, 37, 34] 79
36 GCA [10, 33, 38] 81
37 GCC [12, 35, 38] 85
38 GCG [13, 36, 38] 87
39 GCT [14, 37, 38] 89
40 GGA [15, 33, 39] 87
41 GGC [17, 35, 39] 91
42 GGG [18, 36, 39] 93
43 GGT [19, 37, 39] 95
44 GTA [20, 33, 40] 93
45 GTC [

In [10]:
assign_buckets(kmer_to_binary('CTG'), 3)

[23, 28, 32]

In [5]:
import math

4*math.pow(4, 4)

1024.0

In [7]:
def bucket_kmers(kmers: list[bin], k:int):
    """Returns dictionary with buckets and kmers 

    Args:
        kmers (list[bin]): _description_
        k (int): _description_
    """
    ## intialize bucket
    buckets = {}
    
    for kmer in list(set(kmers)):
        kmer_buckets = assign_buckets(kmer, k, [0]*k, 0) ## assign buckets to kmer O(k)
        for buck in kmer_buckets: ## add these kmers to the bucket O(k)
            kmer_s = binary_to_kmer(kmer, k)
            if buck not in buckets: ## there are definitely more efficient ways to do this, but it is fine for this example
                buckets[buck] = [kmer_s]
            else:
                buckets[buck].append(kmer_s)
                
    overlaps = {bucket: buckets[bucket] for bucket in buckets if len(buckets[bucket])>1}            
    return overlaps, buckets
            
def string_to_kmers(genome:str, k:int):
    """Returns list of binary kmers of length k

    Args:
        genome (str): _description_
        k (int): k
    """
    kmers = []
    for i in range(len(genome)-k+1):
        kmers.append(kmer_to_binary(genome[i:i+k]))
    return kmers
        
k = 10
## testing with two strings with 1 nucleotide difference. Important to note that there will be k shared buckets for a single mutation (in this case 
# going higher than 11 doesn't show that since the mutation is less than that far from the end of the string)
kmers = string_to_kmers("ACTATGAATTTTGCTAGGTAGAAATGGTCCT", k) + string_to_kmers("ACTATGAATATTGCTAGGTAGAAATGGTCCT", k)
overlap, buckets = bucket_kmers(kmers, k)
overlap

{426556: ['ACTATGAATA', 'ACTATGAATT'],
 218774: ['AATTTTGCTA', 'AATATTGCTA'],
 836734: ['TTTGCTAGGT', 'ATTGCTAGGT'],
 1351369: ['CTATGAATAT', 'CTATGAATTT'],
 765484: ['ATGAATATTG', 'ATGAATTTTG'],
 2371309: ['TGAATATTGC', 'TGAATTTTGC'],
 2224812: ['TTTTGCTAGG', 'TATTGCTAGG'],
 2208600: ['TATGAATATT', 'TATGAATTTT'],
 1493166: ['GAATTTTGCT', 'GAATATTGCT'],
 715649: ['ATTTTGCTAG', 'ATATTGCTAG']}

In [8]:
##testing with 2 mutations within k
kmers = string_to_kmers("ACTATGAATTTTGCTAGGTAGAAATGGTCCT", k) + string_to_kmers("ACTATGAATATTGCTAGATAGAAATGGTCCT", k)
overlap, buckets = bucket_kmers(kmers, k)
overlap

{2149666: ['TAGGTAGAAA', 'TAGATAGAAA'],
 508181: ['AGGTAGAAAT', 'AGATAGAAAT'],
 426556: ['ACTATGAATA', 'ACTATGAATT'],
 2579521: ['TTGCTAGATA', 'TTGCTAGGTA'],
 1601103: ['GGTAGAAATG', 'GATAGAAATG'],
 218774: ['AATTTTGCTA', 'AATATTGCTA'],
 1337986: ['CTAGGTAGAA', 'CTAGATAGAA'],
 2428305: ['TGCTAGGTAG', 'TGCTAGATAG'],
 1351369: ['CTATGAATAT', 'CTATGAATTT'],
 688466: ['GTAGAAATGG', 'ATAGAAATGG'],
 765484: ['ATGAATATTG', 'ATGAATTTTG'],
 2371309: ['TGAATATTGC', 'TGAATTTTGC'],
 1747056: ['GCTAGGTAGA', 'GCTAGATAGA'],
 2208600: ['TATGAATATT', 'TATGAATTTT'],
 1493166: ['GAATTTTGCT', 'GAATATTGCT'],
 715649: ['ATTTTGCTAG', 'ATATTGCTAG']}

The next two are for a test I was doing by hand

In [44]:
k=3
kmers = string_to_kmers("ACCGGTTA", k)
overlap, buckets = bucket_kmers(kmers, k)
buckets

[12, 5, 11]
[19, 37, 39]
[24, 37, 40]
[13, 28, 30]
[18, 28, 31]
[20, 41, 48]


{12: ['ACC'],
 5: ['ACC'],
 11: ['ACC'],
 19: ['GGT'],
 37: ['GGT', 'GTT'],
 39: ['GGT'],
 24: ['GTT'],
 40: ['GTT'],
 13: ['CCG'],
 28: ['CCG', 'CGG'],
 30: ['CCG'],
 18: ['CGG'],
 31: ['CGG'],
 20: ['TTA'],
 41: ['TTA'],
 48: ['TTA']}

In [40]:
k=3
kmers = string_to_kmers("GAT", k)
overlap, buckets = bucket_kmers(kmers, k)
buckets

[8, 34, 37]


{8: ['GAT'], 34: ['GAT'], 37: ['GAT']}

In [46]:
k=3
kmers = string_to_kmers("CGG", k)
overlap, buckets = bucket_kmers(kmers, k)
buckets

[18, 28, 31]


{18: ['CGG'], 28: ['CGG'], 31: ['CGG']}

In [48]:
k=3
kmers = string_to_kmers("AACGGTTA", k)
overlap, buckets = bucket_kmers(kmers, k)
buckets

[4, 5, 3]
[13, 7, 11]
[19, 37, 39]
[24, 37, 40]
[18, 28, 31]
[20, 41, 48]


{4: ['AAC'],
 5: ['AAC'],
 3: ['AAC'],
 13: ['ACG'],
 7: ['ACG'],
 11: ['ACG'],
 19: ['GGT'],
 37: ['GGT', 'GTT'],
 39: ['GGT'],
 24: ['GTT'],
 40: ['GTT'],
 18: ['CGG'],
 28: ['CGG'],
 31: ['CGG'],
 20: ['TTA'],
 41: ['TTA'],
 48: ['TTA']}

In [49]:
k=3
kmers = string_to_kmers("TAC", k)
overlap, buckets = bucket_kmers(kmers, k)
buckets

[4, 43, 42]


{4: ['TAC'], 43: ['TAC'], 42: ['TAC']}

In [28]:
from Bio import SeqIO

def read_data(fasta:str):
    """Reads in a fasta file and returns the genome sequence (if segmented, will all be appended together)

    Args:
        fasta (str): location of fasta file
        
    Returns:
        string: sequence of genome within fasta file
    """
    reads = []
    for record in SeqIO.parse(fasta, "fasta"):
        reads.append(str(record.seq))
    
    return reads

In [13]:
reads = read_data("test.fasta")

In [14]:
def get_kmers_from_reads(reads, k):
    kmers = set()
    for read in reads:
        read_kmers = string_to_kmers(read, k)
        for kmer in read_kmers:
            kmers.add(kmer)
    return kmers

k=15
kmers = list(get_kmers_from_reads(reads[:10000], k))

In [15]:
overlap, buckets = bucket_kmers(kmers, k)

In [16]:
len(overlap) 

92977